# Alignment Pipeline — Fixed

Fixes: runtime GPU, numpy conflict, audioop, taaldetectie, Drive copy, device check, checkpoint upgrade.

## 0. Runtime check

⚠️ Zorg dat je runtime op **GPU** staat: `Runtime → Change runtime type → T4 GPU`

(Notebook had `accelerator: TPU` — dat werkt niet met CUDA)

In [1]:
import torch
assert torch.cuda.is_available(), '❌ Geen GPU gevonden! Zet runtime op GPU via Runtime → Change runtime type'
print('✅ GPU:', torch.cuda.get_device_name(0))

✅ GPU: Tesla T4


## 1. Clone repo

In [2]:
import os
if not os.path.exists('/content/Video_Analyzer'):
    !git clone https://github.com/Yi-Star32/Video_Analyzer.git /content/Video_Analyzer
%cd /content/Video_Analyzer

Cloning into '/content/Video_Analyzer'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 55 (delta 13), reused 29 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 63.12 KiB | 994.00 KiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/Video_Analyzer


## 2. Dependencies

Fix volgorde: whisperx eerst (trekt numpy 2.x), daarna niets dat downgradet.
`audioop-lts` werkt niet op Python 3.12 → gefilterd.

In [3]:
# Installeer requirements zonder audioop-lts (niet beschikbaar op Python 3.12)
!grep -v 'audioop-lts' requirements.txt > /tmp/requirements_fixed.txt
!pip install -q -r /tmp/requirements_fixed.txt

grep: requirements.txt: binary file matches


In [4]:
# Installeer whisperx (trekt numpy>=2.1 mee)
!pip install -q git+https://github.com/m-bain/whisperx.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 133.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 93.9 MB/s eta 0:00:00


In [1]:
# Verifieer numpy versie — moet >=2.1.0 zijn voor whisperx
import numpy as np
print('numpy:', np.__version__)
assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 1), \
    f'❌ numpy {np.__version__} te oud voor whisperx — herstart runtime en run cellen opnieuw'
print('✅ numpy OK')

numpy: 2.4.6
✅ numpy OK


## 3. Upgrade Lightning checkpoint (eenmalig, elimineert herhaalde warning)

In [2]:
!python -m lightning.pytorch.utilities.upgrade_checkpoint \
    /usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin 2>/dev/null || true
print('✅ Checkpoint upgrade klaar (of al up-to-date)')

✅ Checkpoint upgrade klaar (of al up-to-date)


## 4. Imports

In [3]:
import sys
import warnings
from pathlib import Path
from tqdm import TqdmWarning

warnings.filterwarnings('ignore', category=TqdmWarning)
warnings.filterwarnings('ignore', message='.*gradient_checkpointing.*')

project_path = '/content/Video_Analyzer'
if project_path not in sys.path:
    sys.path.insert(0, project_path)

from audio_matcher.embedding import AudioEmbeddingPipeline
from audio_matcher.phonemes import PhonemeAligner
from audio_matcher.alignment import build_phoneme_index_from_episodes, run_phoneme_pipeline
from audio_matcher.io import export_audio
print('✅ Imports OK')

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


✅ Imports OK


## 5. Google Drive koppelen + audio naar lokale SSD kopiëren

Drive-reads zijn traag (~50 MB/s). Kopieer audio eenmalig naar `/content/audio/` voor 3-5x snelere verwerking.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = Path('/content/drive/MyDrive/projecten/Video_Analyzer_data')
LOCAL_AUDIO  = Path('/content/audio')
SONG_PATH    = DRIVE_ROOT / 'output/separated/htdemucs/audio/vocals.wav'

assert DRIVE_ROOT.exists(), f'Drive root niet gevonden: {DRIVE_ROOT}'
assert SONG_PATH.exists(),  f'vocals.wav niet gevonden: {SONG_PATH}'
print('✅ Drive OK, song gevonden')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive OK, song gevonden


In [7]:
# Kopieer episodes naar lokale SSD (alleen als nog niet gedaan)
import shutil

DRIVE_EPISODES = DRIVE_ROOT / 'data/episodes_audio'
LOCAL_AUDIO.mkdir(exist_ok=True)

copied = 0
for src in DRIVE_EPISODES.rglob('audio.wav'):
    dst = LOCAL_AUDIO / src.parent.name / 'audio.wav'
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied += 1

files = sorted(LOCAL_AUDIO.rglob('audio.wav'))
print(f'✅ {len(files)} bestanden beschikbaar lokaal ({copied} nieuw gekopieerd)')

✅ 30 bestanden beschikbaar lokaal (20 nieuw gekopieerd)


## 6. Models initialiseren

Fixes:
- `language='en'` → geen detectie per bestand (~60s bespaard per episode)
- `AudioEmbeddingPipeline(device=device)` → Wav2Vec2 op GPU
- `compute_type='float16'` → sneller op moderne GPU

In [8]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device.upper()}')

# FIX: geef device mee aan pipeline zodat Wav2Vec2 ook op GPU draait
pipeline = AudioEmbeddingPipeline(device=device)

# FIX: language='en' voorkomt detectie per bestand
# FIX: compute_type='float16' voor ~2x snelheid op GPU
aligner = PhonemeAligner(
    device=device,
    whisper_model='base',
    language='en',
    compute_type='float16',
)
print('✅ Models geladen')

Device: CUDA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

✅ Models geladen


## 7. Phoneme index bouwen

In [35]:
import time, audio_matcher.alignment as _align_mod, inspect

# Patch embed_audio met timing
_orig_embed = pipeline.embed_audio

def _timed_embed(audio, sr=16000, normalize=True):
    t = time.time()
    result = _orig_embed(audio, sr=sr, normalize=normalize)
    print(f'  embed_audio: {time.time()-t:.2f}s, shape={result.shape}')
    return result

pipeline.embed_audio = _timed_embed

# Test op één episode
from audio_matcher.io import load_audio
audio, sr = _fixed_load_audio(str(files[1]))
from audio_matcher.phonemes import PhonemeAligner
phs = aligner.get_phonemes(str(files[1]))
print(f'Phonemes: {len(phs)}')

# Embed eerste 3 phonemes
from audio_matcher.alignment import _extract_audio_segment
for ph in phs[:3]:
    seg = _extract_audio_segment(audio, sr, ph.start, ph.end)
    print(f'  seg len={len(seg)}, phoneme={ph.phoneme}')
    emb = pipeline.embed_audio(seg, sr=sr, normalize=True)
    print(f'  ✅ emb shape={emb.shape}')

2026-06-02 13:25:56 - whisperx.asr - INFO - Detected language: en (0.82) in first 30s of audio
Phonemes: 9231
  seg len=1600, phoneme=I
  embed_audio: 0.30s, shape=(1, 768)
  ✅ emb shape=(1, 768)
  seg len=1088, phoneme=W
  embed_audio: 0.31s, shape=(1, 768)
  ✅ emb shape=(1, 768)
  seg len=1087, phoneme=A
  embed_audio: 0.32s, shape=(1, 768)
  ✅ emb shape=(1, 768)


In [36]:
import audio_matcher.alignment as _align_mod
import inspect

# Kijk welke load_audio alignment gebruikt
src = inspect.getsource(_align_mod.build_phoneme_index_from_episodes)
# Zoek of load_audio direct aangeroepen wordt of via module
print('load_audio in source:', 'load_audio' in src)
print('current load_audio in module:', _align_mod.load_audio)

load_audio in source: True
current load_audio in module: <function _fixed_load_audio at 0x7df029fa3740>


In [38]:
import audio_matcher.alignment as _align_mod
import numpy as np

# Patch build functie met debug logging
audio, sr = _fixed_load_audio(str(files[0]))
phs = aligner.get_phonemes(str(files[0]))
print(f'Phonemes: {len(phs)}')

success, skip, error = 0, 0, 0
for ph in phs[:20]:
    try:
        seg = _align_mod._extract_audio_segment(audio, sr, ph.start, ph.end)
        if len(seg) < sr * 0.05:
            skip += 1
            continue
        emb = pipeline.embed_audio(seg, sr=sr, normalize=True)
        success += 1
    except Exception as e:
        print(f'  ❌ {ph.phoneme}: {e}')
        error += 1

print(f'success={success}, skip={skip}, error={error}')

2026-06-02 13:30:19 - whisperx.asr - INFO - Detected language: en (0.81) in first 30s of audio
Phonemes: 9375
  embed_audio: 0.30s, shape=(1, 768)
  embed_audio: 0.31s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.01s, shape=(1, 768)
  embed_audio: 0.01s, shape=(1, 768)
  embed_audio: 0.31s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
success=20, skip=0, error=0


In [37]:
# Forceer patch op module-niveau én als attribuut
import audio_matcher.alignment
audio_matcher.alignment.load_audio = _fixed_load_audio

# Rebuild
from audio_matcher.alignment import build_phoneme_index_from_episodes
pindex2 = build_phoneme_index_from_episodes(files[:2], aligner, pipeline)
print(f'entries: {len(pindex2.entries)}')

Building phoneme index from episodes:   0%|          | 0/2 [00:00<?, ?it/s]

2026-06-02 13:28:07 - whisperx.asr - INFO - Detected language: en (0.81) in first 30s of audio
  [audio.wav] 9375 phonemes
  embed_audio: 0.30s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.01s, shape=(1, 768)
  embed_audio: 0.01s, shape=(1, 768)
  embed_audio: 0.31s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shap

Building phoneme index from episodes:   0%|          | 0/2 [01:57<?, ?it/s]


KeyboardInterrupt: 

In [42]:
import gc
import numpy as np
from tqdm import tqdm
from audio_matcher.index import IndexEntry, build_phoneme_index
import audio_matcher.alignment as _align_mod

def _fixed_build_phoneme_index_from_episodes(
    file_paths: list,
    aligner,
    embedder,
    batch_size: int = 64  # Verhoog of verlaag op basis van je VRAM
) -> "PhonemeIndex":
    """Gebatchte versie van build_phoneme_index_from_episodes om CPU/GPU bottlenecks te voorkomen."""
    all_embeddings: list[np.ndarray] = []
    all_entries: list[IndexEntry] = []
    entry_counter = 0

    for path in tqdm(file_paths, desc="Building phoneme index from episodes"):
        try:
            phonemes = aligner.get_phonemes(str(path))
        except Exception as e:
            print(f"  Warning: {path} — phoneme extraction failed: {e}")
            phonemes = None

        n_phonemes = len(phonemes) if phonemes else 0
        # print(f"  [{path.name}] {n_phonemes} phonemes")

        if not phonemes:
            # Fallback code (ongewijzigd, gebruikte al embed_batch)
            print(f"  No phonemes found — {path}, falling back to chunk-based")
            try:
                audio, sr = _align_mod.load_audio(str(path))
                from audio_matcher.chunking import chunk_audio_fixed_overlap
                chunks = chunk_audio_fixed_overlap(audio, sr)
                if not chunks:
                    del audio
                    continue
                embs = embedder.embed_batch(chunks)
                for i, (c, emb) in enumerate(zip(chunks, embs)):
                    all_embeddings.append(emb.reshape(1, -1))
                    all_entries.append(
                        IndexEntry(
                            phoneme=f"CHUNK_{i}",
                            start_time=c.start_time,
                            end_time=c.end_time,
                            word="",
                            audio_idx=entry_counter,
                            source_file=str(path),
                        )
                    )
                    entry_counter += 1
                del audio, chunks
            except Exception as e2:
                print(f"  Warning: {path.name} — fallback also failed: {e2}")
                continue
        else:
            # GEOPTIMALISEERD: Verzamel segmenten en stuur ze in batches naar de embedder
            try:
                audio, sr = _align_mod.load_audio(str(path))

                valid_phonemes = []
                segments = []

                for ph in phonemes:
                    seg = _align_mod._extract_audio_segment(audio, sr, ph.start, ph.end)
                    if len(seg) < sr * 0.05:
                        continue
                    valid_phonemes.append(ph)
                    segments.append(seg)

                if segments:
                    # Genereer embeddings in batches
                    # Als embedder.embed_batch geen lists accepteert, verander segments dan in een HuggingFace-achtige input of np.array indien fixed-length
                    try:
                        embs = embedder.embed_batch(segments)
                    except Exception as e:
                        # Fallback als embed_batch struikelt over variabele lengtes: doe het handmatig per batch
                        embs = []
                        for i in range(0, len(segments), batch_size):
                            batch = segments[i:i+batch_size]
                            # Sommige pipelines vereisen een list, andere een gepadde tensor.
                            # We gaan er vanuit dat embed_batch padding intern regelt zoals in de fallback.
                            for s in batch:
                                embs.append(embedder.embed_audio(s, sr=sr, normalize=True))

                    # Verwerk resultaten
                    for ph, emb in zip(valid_phonemes, embs):
                        all_embeddings.append(emb.reshape(1, -1) if emb.ndim == 1 else emb)
                        all_entries.append(
                            IndexEntry(
                                phoneme=ph.phoneme,
                                start_time=ph.start,
                                end_time=ph.end,
                                word=ph.word,
                                audio_idx=entry_counter,
                                source_file=str(path),
                            )
                        )
                        entry_counter += 1

                del audio, segments, valid_phonemes
            except Exception as e:
                print(f"  Warning: {path.name}: {e}")
                continue

        gc.collect()
        if hasattr(embedder, "device") and "cuda" in str(embedder.device):
            import torch
            torch.cuda.empty_cache()

    all_embs = (
        np.vstack(all_embeddings)
        if all_embeddings
        else np.empty((0, 768), dtype="float32")
    )
    print(f"Built phoneme index from {len(file_paths)} files -> {len(all_entries)} phonemes")
    return build_phoneme_index(all_embs, all_entries)

# Overschrijf de functie in de module
_align_mod.build_phoneme_index_from_episodes = _fixed_build_phoneme_index_from_episodes
print("✅ build_phoneme_index_from_episodes succesvol gepatcht!")

✅ build_phoneme_index_from_episodes succesvol gepatcht!


In [40]:
# Stap 1: patch load_audio in alle modules
import sys, numpy as np, soundfile as _sf
from scipy.signal import resample as scipy_resample
import audio_matcher.alignment as _align_mod
import audio_matcher.io as _io_mod

def _fixed_load_audio(file_path: str, sr: int = 16000):
    audio, orig_sr = _sf.read(file_path, always_2d=True)
    audio = audio.mean(axis=1).astype(np.float32)
    if orig_sr != sr:
        n_samples = int(len(audio) * sr / orig_sr)
        audio = scipy_resample(audio, n_samples).astype(np.float32)
    return audio, sr

_align_mod.load_audio = _fixed_load_audio
_io_mod.load_audio = _fixed_load_audio

# Stap 2: test patch op één bestand
test_audio, test_sr = _fixed_load_audio(str(files[0]))
print(f'✅ load_audio OK: shape={test_audio.shape}, sr={test_sr}')

# Stap 3: herbouw index
from audio_matcher.alignment import build_phoneme_index_from_episodes
pindex = build_phoneme_index_from_episodes(files, aligner, pipeline)
print(f'✅ Index entries: {len(pindex.entries)}')

✅ load_audio OK: shape=(21473628,), sr=16000


Building phoneme index from episodes:   0%|          | 0/30 [00:00<?, ?it/s]

2026-06-02 13:40:14 - whisperx.asr - INFO - Detected language: en (0.81) in first 30s of audio
  [audio.wav] 9375 phonemes



  0%|          | 0/348 [00:00<?, ?it/s]


  embed_audio: 0.31s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.01s, shape=(1, 768)
  embed_audio: 0.01s, shape=(1, 768)
  embed_audio: 0.31s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
 

Building phoneme index from episodes:   0%|          | 0/30 [02:04<?, ?it/s]


KeyboardInterrupt: 

In [13]:
# Fix: vervang librosa.load met soundfile+resampy (geen numba dependency)
!pip install -q soundfile resampy

io_path = '/content/Video_Analyzer/audio_matcher/io.py'
with open(io_path, 'r') as f:
    src = f.read()

patched = src.replace(
    'import librosa',
    'import soundfile as _sf\nimport resampy as _resampy'
).replace(
    'audio, sr = librosa.load(file_path, sr=sr, mono=True)',
    '''audio, _orig_sr = _sf.read(file_path, always_2d=True)
    audio = audio.mean(axis=1)  # mono
    import numpy as np
    audio = audio.astype(np.float32)
    if _orig_sr != sr:
        audio = _resampy.resample(audio, _orig_sr, sr)'''
)

with open(io_path, 'w') as f:
    f.write(patched)
print('✅ io.py gepatcht — librosa vervangen door soundfile+resampy')

✅ io.py gepatcht — librosa vervangen door soundfile+resampy


In [44]:
import gc
import numpy as np
from tqdm import tqdm
from audio_matcher.index import IndexEntry, build_phoneme_index
import audio_matcher.alignment as _align_mod

def _fixed_build_phoneme_index_from_episodes(
    file_paths: list,
    aligner,
    embedder,
    batch_size: int = 32  # Veilig en snel voor GPU
) -> "PhonemeIndex":
    all_embeddings: list[np.ndarray] = []
    all_entries: list[IndexEntry] = []
    entry_counter = 0

    for path in tqdm(file_paths, desc="Building phoneme index from episodes"):
        try:
            phonemes = aligner.get_phonemes(str(path))
        except Exception as e:
            print(f"  Warning: {path} — phoneme extraction failed: {e}")
            phonemes = None

        n_phonemes = len(phonemes) if phonemes else 0
        print(f"  [{path.name}] {n_phonemes} phonemes")

        if not phonemes:
            # Fallback code (ongewijzigd)
            try:
                audio, sr = _align_mod.load_audio(str(path))
                from audio_matcher.chunking import chunk_audio_fixed_overlap
                chunks = chunk_audio_fixed_overlap(audio, sr)
                if not chunks:
                    del audio
                    continue
                embs = embedder.embed_batch(chunks)
                for i, (c, emb) in enumerate(zip(chunks, embs)):
                    all_embeddings.append(emb.reshape(1, -1))
                    all_entries.append(IndexEntry(phoneme=f"CHUNK_{i}", start_time=c.start_time, end_time=c.end_time, word="", audio_idx=entry_counter, source_file=str(path)))
                    entry_counter += 1
                del audio, chunks
            except Exception:
                continue
        else:
            try:
                audio, sr = _align_mod.load_audio(str(path))

                valid_phonemes = []
                segments = []

                for ph in phonemes:
                    seg = _align_mod._extract_audio_segment(audio, sr, ph.start, ph.end)
                    if len(seg) < sr * 0.05:
                        continue
                    valid_phonemes.append(ph)
                    segments.append(seg)

                if segments:
                    print(f"  -> Processing {len(segments)} valid segments in batches of {batch_size}...")

                    # Hak de lijst handmatig in harde batches om de pipeline te ontlasten
                    for i in range(0, len(segments), batch_size):
                        batch_seg = segments[i:i+batch_size]
                        batch_ph = valid_phonemes[i:i+batch_size]

                        # We gebruiken hier embed_batch per kleine sub-lijst
                        try:
                            batch_embs = embedder.embed_batch(batch_seg)
                        except Exception:
                            # Ultieme fallback per foneem als embed_batch echt niet werkt met lists
                            batch_embs = [embedder.embed_audio(s, sr=sr, normalize=True) for s in batch_seg]

                        for ph, emb in zip(batch_ph, batch_embs):
                            all_embeddings.append(emb.reshape(1, -1) if emb.ndim == 1 else emb)
                            all_entries.append(
                                IndexEntry(
                                    phoneme=ph.phoneme,
                                    start_time=ph.start,
                                    end_time=ph.end,
                                    word=ph.word,
                                    audio_idx=entry_counter,
                                    source_file=str(path),
                                )
                            )
                            entry_counter += 1

                del audio, segments, valid_phonemes
            except Exception as e:
                print(f"  Warning: {path.name}: {e}")
                continue

        gc.collect()
        if hasattr(embedder, "device") and "cuda" in str(embedder.device):
            import torch
            torch.cuda.empty_cache()

    all_embs = np.vstack(all_embeddings) if all_embeddings else np.empty((0, 768), dtype="float32")
    return build_phoneme_index(all_embs, all_entries)

_align_mod.build_phoneme_index_from_episodes = _fixed_build_phoneme_index_from_episodes
print("✅ Nieuwe batch-patch toegepast!")

✅ Nieuwe batch-patch toegepast!


In [45]:
if not files:
    raise RuntimeError('Geen audio bestanden gevonden')

pindex = build_phoneme_index_from_episodes(files, aligner, pipeline)
print('✅ Phoneme index klaar')

Building phoneme index from episodes:   0%|          | 0/30 [00:00<?, ?it/s]

2026-06-02 13:55:38 - whisperx.asr - INFO - Detected language: en (0.81) in first 30s of audio
  [audio.wav] 9375 phonemes



  0%|          | 0/348 [00:00<?, ?it/s]


  embed_audio: 0.30s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.31s, shape=(1, 768)
  embed_audio: 0.01s, shape=(1, 768)
  embed_audio: 0.01s, shape=(1, 768)
  embed_audio: 0.31s, shape=(1, 768)
  embed_audio: 0.31s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
  embed_audio: 0.32s, shape=(1, 768)
 

Building phoneme index from episodes:   0%|          | 0/30 [03:07<?, ?it/s]


KeyboardInterrupt: 

## 8. Pipeline uitvoeren + exporteren

In [23]:
import pathlib, re, sys

# Zoek numba __init__.py zonder te importeren
import importlib.util
spec = importlib.util.find_spec('numba')
nb_init = pathlib.Path(spec.origin)
print('Patching:', nb_init)

src = nb_init.read_text()
patched = src.replace(
    'raise ImportError(msg)',
    'pass  # patched: numpy check disabled'
)
nb_init.write_text(patched)

# Verwijder pyc cache
for pyc in nb_init.parent.rglob('*.pyc'):
    pyc.unlink(missing_ok=True)

print('✅ gepatcht — run nu de pipeline cel direct (geen herstart nodig)')

Patching: /usr/local/lib/python3.12/dist-packages/numba/__init__.py
✅ gepatcht — run nu de pipeline cel direct (geen herstart nodig)


In [25]:
# Blokkeer librosa permanent via sys.modules mock
import sys
from unittest.mock import MagicMock

# Maak nep-librosa die nooit numba aanraakt
mock_librosa = MagicMock()
sys.modules['librosa'] = mock_librosa
sys.modules['librosa.core'] = mock_librosa
sys.modules['librosa.core.audio'] = mock_librosa

# Verifieer dat io.py's eigen load_audio werkt zonder librosa
from importlib import reload
import audio_matcher.io as _io_mod
reload(_io_mod)

from audio_matcher.io import load_audio
import numpy as np
test_arr, test_sr = load_audio(str(list(__import__('pathlib').Path('/content/audio').rglob('audio.wav'))[0]))
print(f'✅ load_audio werkt: shape={test_arr.shape}, sr={test_sr}')

✅ load_audio werkt: shape=(21478272,), sr=16000


In [27]:
import sys, numpy as np
import soundfile as _sf
import resampy as _resampy
import audio_matcher.io as _io_mod

def _fixed_load_audio(file_path: str, sr: int = 16000):
    audio, orig_sr = _sf.read(file_path, always_2d=True)
    audio = audio.mean(axis=1).astype(np.float32)
    if orig_sr != sr:
        audio = _resampy.resample(audio, orig_sr, sr)
    return audio, sr

# Patch zowel de module als de al-geïmporteerde referentie in alignment
_io_mod.load_audio = _fixed_load_audio

import audio_matcher.alignment as _align_mod
_align_mod.load_audio = _fixed_load_audio

print('✅ load_audio gepatcht in geheugen')

✅ load_audio gepatcht in geheugen


In [29]:
import sys, numpy as np, soundfile as _sf
from scipy.signal import resample as scipy_resample
import audio_matcher.alignment as _align_mod

def _fixed_load_audio(file_path: str, sr: int = 16000):
    audio, orig_sr = _sf.read(file_path, always_2d=True)
    audio = audio.mean(axis=1).astype(np.float32)
    if orig_sr != sr:
        n_samples = int(len(audio) * sr / orig_sr)
        audio = scipy_resample(audio, n_samples).astype(np.float32)
    return audio, sr

_align_mod.load_audio = _fixed_load_audio
print('✅ load_audio gepatcht — scipy resample, geen numba')

✅ load_audio gepatcht — scipy resample, geen numba


In [30]:
OUTPUT_PATH = str(DRIVE_ROOT / 'output/aligned_output_colab3.wav')

final_audio = run_phoneme_pipeline(SONG_PATH, None, aligner, pipeline, pindex=pindex)
print(f'Output lengte: {len(final_audio)} ms')

export_audio(final_audio, OUTPUT_PATH)
print(f'✅ Opgeslagen: {OUTPUT_PATH}')

2026-06-02 12:59:49 - whisperx.asr - INFO - Detected language: en (0.84) in first 30s of audio
song phonemes: 1981, matched: 0
output_ms: 220171
Output lengte: 220171 ms
✅ Opgeslagen: /content/drive/MyDrive/projecten/Video_Analyzer_data/output/aligned_output_colab2.wav


In [31]:
from pydub import AudioSegment
seg = AudioSegment.from_wav('/content/drive/MyDrive/projecten/Video_Analyzer_data/output/aligned_output_colab2.wav')
print(f'Duur: {len(seg)/1000:.1f}s, max dBFS: {seg.max_dBFS:.1f}')

Duur: 220.2s, max dBFS: -inf


In [33]:
# Check wat alignment.py doet met de entries
import audio_matcher.alignment as _align_mod
import inspect
print(inspect.getsource(_align_mod.build_phoneme_index_from_episodes))

def build_phoneme_index_from_episodes(
    file_paths: List[Path],
    aligner: PhonemeAligner,
    embedder: AudioEmbeddingPipeline,
) -> PhonemeIndex:
    """Build a combined phoneme FAISS index from multiple episode audio files.

    Each phoneme entry tracks its source file so stitching can lazy-load
    the correct audio later.
    """
    import gc
    from tqdm import tqdm

    all_embeddings: list[np.ndarray] = []
    all_entries: list[IndexEntry] = []
    entry_counter = 0

    for path in tqdm(file_paths, desc="Building phoneme index from episodes"):
        try:
            phonemes = aligner.get_phonemes(str(path))
        except Exception as e:
            print(f"  Warning: {path} — phoneme extraction failed: {e}")
            phonemes = None

        n_phonemes = len(phonemes) if phonemes else 0
        print(f"  [{path.name}] {n_phonemes} phonemes")

        if not phonemes:
            print(f"  No phonemes found — {path}, falling back to chunk-based")
            try: